In [ ]:
# importing necessary libraries
import pandas as pd
import numpy as np
from datetime import datetime, timezone


In [3]:
# OMS order data 
orders = pd.read_csv("../output/orders.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"])


In [4]:
# normaize cod and rto signals 
orders["is_cod"] = orders["payment_type"].eq("COD").astype(int)
orders["is_rto"] = orders["delivery_status"].eq("RTO").astype(int)


In [5]:
agg = (
    orders.groupby(["sku_id", "warehouse_id"])
    .agg(
        total_orders=("order_date", "count"),
        cod_orders=("is_cod", "sum"),
        rto_orders=("is_rto", "sum"),
    )
    .reset_index()
)


In [6]:
# COd metrics 
agg["cod_share"] = agg["cod_orders"] / agg["total_orders"]
agg["cod_rto_rate"] = np.where(
    agg["cod_orders"] > 0,
    agg["rto_orders"] / agg["cod_orders"],
    0.0,
)

agg["cod_success_rate"] = 1 - agg["cod_rto_rate"]


In [ ]:
# define sufficient data flag
MIN_COD_ORDERS = 30

agg["cod_data_sufficient"] = agg["cod_orders"] >= MIN_COD_ORDERS


In [ ]:
# COD risk bucket assignment
def cod_risk_bucket(row):
    if not row["cod_data_sufficient"]:
        return "MEDIUM"  # default cautious stance

    if row["cod_rto_rate"] >= 0.35:
        return "HIGH"
    elif row["cod_rto_rate"] >= 0.2:
        return "MEDIUM"
    else:
        return "LOW"

agg["cod_risk_bucket"] = agg.apply(cod_risk_bucket, axis=1)


In [9]:
RISK_TO_ACTION = {
    "LOW": "ALLOW_COD",
    "MEDIUM": "LIMIT_COD",
    "HIGH": "DISABLE_COD",
}

agg["cod_policy_action"] = agg["cod_risk_bucket"].map(RISK_TO_ACTION)


In [10]:
# financial risk flag
agg["financial_risk_flag"] = (
    (agg["cod_risk_bucket"] == "HIGH") &
    (agg["cod_share"] > 0.4)
)


In [11]:
RUN_DATE = datetime.now(timezone.utc).date()

final_cod = agg[[
    "sku_id",
    "warehouse_id",
    "cod_share",
    "cod_rto_rate",
    "cod_success_rate",
    "cod_risk_bucket",
    "cod_policy_action",
    "financial_risk_flag",
]]

final_cod["run_date"] = RUN_DATE
final_cod.head(20)


C:\Users\ACER\AppData\Local\Temp\ipykernel_4524\3731100137.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_cod["run_date"] = RUN_DATE


,sku_id,warehouse_id,cod_share,cod_rto_rate,cod_success_rate,cod_risk_bucket,cod_policy_action,financial_risk_flag,run_date
0,SKU0001,east,0.445104,0.580000,0.420000,HIGH,DISABLE_COD,True,2025-12-26
1,SKU0001,north,0.446262,0.579058,0.420942,HIGH,DISABLE_COD,True,2025-12-26
2,SKU0001,south,0.452694,0.566000,0.434000,HIGH,DISABLE_COD,True,2025-12-26
3,SKU0001,west,0.455299,0.533525,0.466475,HIGH,DISABLE_COD,True,2025-12-26
4,SKU0002,east,0.461161,0.541667,0.458333,HIGH,DISABLE_COD,True,2025-12-26
5,SKU0002,north,0.452988,0.553210,0.446790,HIGH,DISABLE_COD,True,2025-12-26
6,SKU0002,south,0.447840,0.567789,0.432211,HIGH,DISABLE_COD,True,2025-12-26
7,SKU0002,west,0.448361,0.592322,0.407678,HIGH,DISABLE_COD,True,2025-12-26
8,SKU0003,east,0.442837,0.562208,0.437792,HIGH,DISABLE_COD,True,2025-12-26
9,SKU0003,north,0.454670,0.541761,0.458239,HIGH,DISABLE_COD,True,2025-12-26


In [12]:
final_cod.to_csv(
    "../artifacts/cod_intelligence.csv",
    index=False
)

print("Saved cod_intelligence.csv")


Saved cod_intelligence.csv
